# Part B: WGAN and WGAN-GP on CIFAR-10

**Assignment:** Image Generation using Generative Models  
**Dataset:** CIFAR-10  
**Framework:** PyTorch  
**Part:** B only (WGAN & WGAN-GP)

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from torchvision.models import inception_v3
from scipy import linalg

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Hyperparameters
BATCH_SIZE = 64
IMAGE_SIZE = 32
CHANNELS = 3
LATENT_DIM = 128
EPOCHS = 50  # Adjust based on compute budget; 50 is reasonable for CIFAR-10
LR = 2e-4
BETA1 = 0.5
BETA2 = 0.999
N_CRITIC = 5          # Number of discriminator updates per generator update
CLIP_VALUE = 0.01     # Weight clipping for WGAN
LAMBDA_GP = 10        # Gradient penalty coefficient for WGAN-GP
SAMPLE_INTERVAL = 5   # Epoch interval for generating samples
FID_INTERVAL = 10     # Epoch interval for computing FID

# Create output directories
os.makedirs('wgan_samples', exist_ok=True)
os.makedirs('wgan_gp_samples', exist_ok=True)
os.makedirs('plots', exist_ok=True)

## 1. Data Loading & Preprocessing

CIFAR-10 images are normalized to **[0, 1]** as specified.

In [ ]:
# CIFAR-10 preprocessing: normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL [0,255] -> Tensor [0,1]
])

# Load CIFAR-10
train_dataset = datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of batches: {len(train_loader)}")

# Visualize a batch of real images
def show_real_images(dataloader, num_images=64):
    real_batch = next(iter(dataloader))
    images = real_batch[0][:num_images]
    grid = utils.make_grid(images, nrow=8, normalize=True, value_range=(0, 1))
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.title('Real CIFAR-10 Images')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('plots/real_images.png', dpi=150)
    plt.show()

show_real_images(train_loader)

## 2. Model Architectures

### Generator (for both WGAN and WGAN-GP)
A standard DCGAN-style generator that maps a latent vector $z \sim \mathcal{N}(0, I)$ to a $32 \times 32 \times 3$ image.

### Discriminator / Critic
A DCGAN-style critic that outputs a scalar score (no sigmoid for WGAN/WGAN-GP).

In [ ]:
class Generator(nn.Module):
    """
    DCGAN-style Generator for CIFAR-10 (32x32).
    Maps latent vector z -> image.
    """
    def __init__(self, latent_dim=128, channels=3):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim

        self.main = nn.Sequential(
            # Input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # State: 512 x 4 x 4

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # State: 256 x 8 x 8

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # State: 128 x 16 x 16

            nn.ConvTranspose2d(128, channels, 4, 2, 1, bias=False),
            nn.Tanh()  # Output in [-1, 1]
            # State: 3 x 32 x 32
        )

    def forward(self, z):
        z = z.view(-1, self.latent_dim, 1, 1)
        return self.main(z)


class Critic(nn.Module):
    """
    DCGAN-style Critic (Discriminator without sigmoid).
    Outputs a scalar score for Wasserstein distance estimation.
    """
    def __init__(self, channels=3):
        super(Critic, self).__init__()

        self.main = nn.Sequential(
            # Input: 3 x 32 x 32
            nn.Conv2d(channels, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # State: 128 x 16 x 16

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(256, affine=True),  # LayerNorm/InstanceNorm preferred for WGAN
            nn.LeakyReLU(0.2, inplace=True),
            # State: 256 x 8 x 8

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(512, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            # State: 512 x 4 x 4

            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
            # Output: 1 x 1 x 1
        )

    def forward(self, x):
        return self.main(x).view(-1)


# Initialize models
generator_wgan = Generator(LATENT_DIM, CHANNELS).to(DEVICE)
critic_wgan = Critic(CHANNELS).to(DEVICE)

generator_wgan_gp = Generator(LATENT_DIM, CHANNELS).to(DEVICE)
critic_wgan_gp = Critic(CHANNELS).to(DEVICE)

print(f"Generator parameters: {sum(p.numel() for p in generator_wgan.parameters()):,}")
print(f"Critic parameters: {sum(p.numel() for p in critic_wgan.parameters()):,}")

In [ ]:
def weights_init(m):
    """Initialize network weights."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Apply to all models
generator_wgan.apply(weights_init)
critic_wgan.apply(weights_init)
generator_wgan_gp.apply(weights_init)
critic_wgan_gp.apply(weights_init)

print("Weights initialized.")

## 3. FID (Fréchet Inception Distance) Computation

FID measures the similarity between generated and real image distributions using InceptionV3 feature statistics. Lower is better.

In [ ]:
class InceptionFeatureExtractor(nn.Module):
    """
    Extracts 2048-dim features from InceptionV3 for FID computation.
    Uses the pool3 layer output.
    """
    def __init__(self):
        super(InceptionFeatureExtractor, self).__init__()
        inception = inception_v3(pretrained=True, transform_input=False)
        self.blocks = nn.ModuleList([
            inception.Conv2d_1a_3x3,
            inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(3, stride=2),
            inception.Conv2d_3b_1x1,
            inception.Conv2d_4a_3x3,
            nn.MaxPool2d(3, stride=2),
            inception.Mixed_5b,
            inception.Mixed_5c,
            inception.Mixed_5d,
            inception.Mixed_6a,
            inception.Mixed_6b,
            inception.Mixed_6c,
            inception.Mixed_6d,
            inception.Mixed_6e,
            inception.Mixed_7a,
            inception.Mixed_7b,
            inception.Mixed_7c,
        ])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        # Inception expects images in [-1, 1] or [0, 1] with transform_input
        # Our images are in [0, 1], we scale to [-1, 1]
        x = 2 * x - 1
        for block in self.blocks:
            x = block(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x


def calculate_fid(real_images, fake_images, batch_size=50, device=DEVICE):
    """
    Calculate FID between real and fake image tensors.
    Both should be in [0, 1] range.
    """
    inception = InceptionFeatureExtractor().to(device)
    inception.eval()

    def get_features(images):
        features = []
        with torch.no_grad():
            for i in range(0, len(images), batch_size):
                batch = images[i:i+batch_size].to(device)
                # Resize to 299x299 for Inception
                batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
                feat = inception(batch)
                features.append(feat.cpu().numpy())
        return np.concatenate(features, axis=0)

    real_features = get_features(real_images)
    fake_features = get_features(fake_images)

    # Calculate mean and covariance
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)

    mu_fake = np.mean(fake_features, axis=0)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Calculate FID
    diff = mu_real - mu_fake
    covmean, _ = linalg.sqrtm(sigma_real @ sigma_fake, disp=False)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return float(fid)


# Prepare a fixed set of real images for FID computation
print("Preparing real images for FID...")
real_images_for_fid = []
for i, (imgs, _) in enumerate(test_loader):
    real_images_for_fid.append(imgs)
    if len(real_images_for_fid) * BATCH_SIZE >= 10000:
        break
real_images_for_fid = torch.cat(real_images_for_fid, dim=0)[:10000]
print(f"Real images for FID: {real_images_for_fid.shape}")

## 4. WGAN Training (with Weight Clipping)

**Key differences from standard GAN:**
- Critic outputs unbounded scalar (no sigmoid)
- Uses **Wasserstein loss**: maximize $\mathbb{E}[D(x)] - \mathbb{E}[D(G(z))]$
- **Weight clipping** enforces Lipschitz constraint: $w \in [-c, c]$
- More critic updates ($n_{critic}=5$) per generator update

In [ ]:
def train_wgan(generator, critic, epochs, latent_dim, dataloader, device, 
               lr=LR, n_critic=N_CRITIC, clip_value=CLIP_VALUE):
    """
    Train WGAN with weight clipping.
    Returns training history.
    """
    optimizer_G = optim.RMSprop(generator.parameters(), lr=lr)
    optimizer_D = optim.RMSprop(critic.parameters(), lr=lr)

    history = {'g_loss': [], 'd_loss': [], 'fid': [], 'epoch': []}

    fixed_noise = torch.randn(64, latent_dim, device=device)

    print(f"\n{'='*60}")
    print("TRAINING WGAN (Weight Clipping)")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        num_batches = 0

        pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{epochs}')
        for i, (real_imgs, _) in enumerate(pbar):
            batch_size = real_imgs.size(0)
            real_imgs = real_imgs.to(device)

            # ---------------------
            #  Train Critic
            # ---------------------
            optimizer_D.zero_grad()

            # Real images
            real_validity = critic(real_imgs)

            # Fake images
            z = torch.randn(batch_size, latent_dim, device=device)
            fake_imgs = generator(z).detach()
            fake_validity = critic(fake_imgs)

            # Wasserstein loss for critic: maximize E[D(x)] - E[D(G(z))]
            # = minimize -(E[D(x)] - E[D(G(z))])
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity)
            d_loss.backward()
            optimizer_D.step()

            # Weight clipping
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

            epoch_d_loss += d_loss.item()

            # ---------------------
            #  Train Generator
            # ---------------------
            if i % n_critic == 0:
                optimizer_G.zero_grad()

                z = torch.randn(batch_size, latent_dim, device=device)
                gen_imgs = generator(z)
                gen_validity = critic(gen_imgs)

                # Wasserstein loss for generator: maximize E[D(G(z))]
                # = minimize -E[D(G(z))]
                g_loss = -torch.mean(gen_validity)
                g_loss.backward()
                optimizer_G.step()

                epoch_g_loss += g_loss.item()

            num_batches += 1
            pbar.set_postfix({'D_loss': f'{d_loss.item():.4f}', 'G_loss': f'{g_loss.item() if i % n_critic == 0 else 0:.4f}'})

        avg_g_loss = epoch_g_loss / (num_batches // n_critic) if num_batches // n_critic > 0 else 0
        avg_d_loss = epoch_d_loss / num_batches

        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)
        history['epoch'].append(epoch)

        print(f"\n[Epoch {epoch}/{epochs}] D_loss: {avg_d_loss:.4f} | G_loss: {avg_g_loss:.4f}")

        # Save sample images
        if epoch % SAMPLE_INTERVAL == 0 or epoch == 1:
            generator.eval()
            with torch.no_grad():
                fake_samples = generator(fixed_noise)
                # Map from [-1, 1] to [0, 1] for visualization
                fake_samples = (fake_samples + 1) / 2
                grid = utils.make_grid(fake_samples, nrow=8, normalize=False)
                plt.figure(figsize=(8, 8))
                plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
                plt.title(f'WGAN Samples - Epoch {epoch}')
                plt.axis('off')
                plt.tight_layout()
                plt.savefig(f'wgan_samples/epoch_{epoch:03d}.png', dpi=150)
                plt.show()
            generator.train()

        # Compute FID
        if epoch % FID_INTERVAL == 0 or epoch == epochs:
            generator.eval()
            with torch.no_grad():
                fake_images = []
                for _ in range(10000 // BATCH_SIZE):
                    z = torch.randn(BATCH_SIZE, latent_dim, device=device)
                    imgs = generator(z)
                    imgs = (imgs + 1) / 2  # [-1,1] -> [0,1]
                    fake_images.append(imgs.cpu())
                fake_images = torch.cat(fake_images, dim=0)[:10000]

            fid = calculate_fid(real_images_for_fid, fake_images)
            history['fid'].append(fid)
            print(f">>> FID Score: {fid:.2f}")
            generator.train()

    return history

# Train WGAN
history_wgan = train_wgan(
    generator=generator_wgan,
    critic=critic_wgan,
    epochs=EPOCHS,
    latent_dim=LATENT_DIM,
    dataloader=train_loader,
    device=DEVICE
)

## 5. WGAN-GP Training (with Gradient Penalty)

**Improvements over WGAN:**
- Replaces weight clipping with **gradient penalty** for smoother Lipschitz constraint
- Penalty: $\lambda \mathbb{E}[(||\nabla_{\hat{x}} D(\hat{x})||_2 - 1)^2]$ where $\hat{x} = \epsilon x + (1-\epsilon)G(z)$
- Better training stability and sample quality

In [ ]:
def compute_gradient_penalty(critic, real_imgs, fake_imgs, device):
    """
    Compute gradient penalty for WGAN-GP.
    """
    batch_size = real_imgs.size(0)

    # Random interpolation
    epsilon = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolates = epsilon * real_imgs + (1 - epsilon) * fake_imgs
    interpolates = interpolates.requires_grad_(True)

    # Critic output for interpolates
    d_interpolates = critic(interpolates)

    # Gradients w.r.t. interpolates
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    # Flatten gradients
    gradients = gradients.view(batch_size, -1)

    # Gradient norm
    gradient_norm = gradients.norm(2, dim=1)

    # Penalty: (||grad|| - 1)^2
    penalty = ((gradient_norm - 1) ** 2).mean()
    return penalty


def train_wgan_gp(generator, critic, epochs, latent_dim, dataloader, device,
                  lr=LR, n_critic=N_CRITIC, lambda_gp=LAMBDA_GP):
    """
    Train WGAN-GP with gradient penalty.
    Returns training history.
    """
    optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(BETA1, BETA2))
    optimizer_D = optim.Adam(critic.parameters(), lr=lr, betas=(BETA1, BETA2))

    history = {'g_loss': [], 'd_loss': [], 'fid': [], 'epoch': []}

    fixed_noise = torch.randn(64, latent_dim, device=device)

    print(f"\n{'='*60}")
    print("TRAINING WGAN-GP (Gradient Penalty)")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        num_batches = 0

        pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{epochs}')
        for i, (real_imgs, _) in enumerate(pbar):
            batch_size = real_imgs.size(0)
            real_imgs = real_imgs.to(device)

            # ---------------------
            #  Train Critic
            # ---------------------
            optimizer_D.zero_grad()

            # Real images
            real_validity = critic(real_imgs)

            # Fake images
            z = torch.randn(batch_size, latent_dim, device=device)
            fake_imgs = generator(z).detach()
            fake_validity = critic(fake_imgs)

            # Gradient penalty
            gp = compute_gradient_penalty(critic, real_imgs, fake_imgs, device)

            # WGAN-GP loss: -E[D(x)] + E[D(G(z))] + lambda * GP
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gp
            d_loss.backward()
            optimizer_D.step()

            epoch_d_loss += d_loss.item()

            # ---------------------
            #  Train Generator
            # ---------------------
            if i % n_critic == 0:
                optimizer_G.zero_grad()

                z = torch.randn(batch_size, latent_dim, device=device)
                gen_imgs = generator(z)
                gen_validity = critic(gen_imgs)

                # Generator loss: -E[D(G(z))]
                g_loss = -torch.mean(gen_validity)
                g_loss.backward()
                optimizer_G.step()

                epoch_g_loss += g_loss.item()

            num_batches += 1
            pbar.set_postfix({'D_loss': f'{d_loss.item():.4f}', 'G_loss': f'{g_loss.item() if i % n_critic == 0 else 0:.4f}'})

        avg_g_loss = epoch_g_loss / (num_batches // n_critic) if num_batches // n_critic > 0 else 0
        avg_d_loss = epoch_d_loss / num_batches

        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)
        history['epoch'].append(epoch)

        print(f"\n[Epoch {epoch}/{epochs}] D_loss: {avg_d_loss:.4f} | G_loss: {avg_g_loss:.4f}")

        # Save sample images
        if epoch % SAMPLE_INTERVAL == 0 or epoch == 1:
            generator.eval()
            with torch.no_grad():
                fake_samples = generator(fixed_noise)
                fake_samples = (fake_samples + 1) / 2
                grid = utils.make_grid(fake_samples, nrow=8, normalize=False)
                plt.figure(figsize=(8, 8))
                plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
                plt.title(f'WGAN-GP Samples - Epoch {epoch}')
                plt.axis('off')
                plt.tight_layout()
                plt.savefig(f'wgan_gp_samples/epoch_{epoch:03d}.png', dpi=150)
                plt.show()
            generator.train()

        # Compute FID
        if epoch % FID_INTERVAL == 0 or epoch == epochs:
            generator.eval()
            with torch.no_grad():
                fake_images = []
                for _ in range(10000 // BATCH_SIZE):
                    z = torch.randn(BATCH_SIZE, latent_dim, device=device)
                    imgs = generator(z)
                    imgs = (imgs + 1) / 2
                    fake_images.append(imgs.cpu())
                fake_images = torch.cat(fake_images, dim=0)[:10000]

            fid = calculate_fid(real_images_for_fid, fake_images)
            history['fid'].append(fid)
            print(f">>> FID Score: {fid:.2f}")
            generator.train()

    return history

# Train WGAN-GP
history_wgan_gp = train_wgan_gp(
    generator=generator_wgan_gp,
    critic=critic_wgan_gp,
    epochs=EPOCHS,
    latent_dim=LATENT_DIM,
    dataloader=train_loader,
    device=DEVICE
)

## 6. Training Curves & Comparison

In [ ]:
# Plot training losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# WGAN Losses
axes[0].plot(history_wgan['epoch'], history_wgan['d_loss'], label='Critic Loss', color='blue')
axes[0].plot(history_wgan['epoch'], history_wgan['g_loss'], label='Generator Loss', color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('WGAN Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# WGAN-GP Losses
axes[1].plot(history_wgan_gp['epoch'], history_wgan_gp['d_loss'], label='Critic Loss', color='blue')
axes[1].plot(history_wgan_gp['epoch'], history_wgan_gp['g_loss'], label='Generator Loss', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('WGAN-GP Training Losses')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/training_losses.png', dpi=150)
plt.show()

# Print final FID scores
print(f"\n{'='*60}")
print("FINAL FID SCORES")
print(f"{'='*60}")
if history_wgan['fid']:
    print(f"WGAN    Final FID: {history_wgan['fid'][-1]:.2f}")
if history_wgan_gp['fid']:
    print(f"WGAN-GP Final FID: {history_wgan_gp['fid'][-1]:.2f}")
print(f"{'='*60}")

## 7. Generate Random Samples

Generate and visualize 100 random samples from both trained models for Part C comparative analysis.

In [ ]:
def generate_samples(generator, num_samples=100, latent_dim=LATENT_DIM, device=DEVICE):
    """Generate random samples from the generator."""
    generator.eval()
    samples = []
    with torch.no_grad():
        for _ in range(num_samples // BATCH_SIZE + 1):
            z = torch.randn(BATCH_SIZE, latent_dim, device=device)
            imgs = generator(z)
            imgs = (imgs + 1) / 2  # [-1,1] -> [0,1]
            samples.append(imgs.cpu())
    samples = torch.cat(samples, dim=0)[:num_samples]
    return samples


# Generate 100 samples from WGAN
wgan_samples = generate_samples(generator_wgan, num_samples=100)
grid_wgan = utils.make_grid(wgan_samples, nrow=10, normalize=False)

plt.figure(figsize=(12, 12))
plt.imshow(grid_wgan.permute(1, 2, 0).numpy())
plt.title('WGAN: 100 Random Samples', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('plots/wgan_100_samples.png', dpi=150)
plt.show()

# Generate 100 samples from WGAN-GP
wgan_gp_samples = generate_samples(generator_wgan_gp, num_samples=100)
grid_wgan_gp = utils.make_grid(wgan_gp_samples, nrow=10, normalize=False)

plt.figure(figsize=(12, 12))
plt.imshow(grid_wgan_gp.permute(1, 2, 0).numpy())
plt.title('WGAN-GP: 100 Random Samples', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('plots/wgan_gp_100_samples.png', dpi=150)
plt.show()

print(f"Generated 100 samples from WGAN and WGAN-GP.")
print(f"Saved to: plots/wgan_100_samples.png, plots/wgan_gp_100_samples.png")

## 8. Save Trained Models

In [ ]:
# Save model checkpoints
torch.save({
    'generator': generator_wgan.state_dict(),
    'critic': critic_wgan.state_dict(),
    'history': history_wgan,
}, 'wgan_checkpoint.pth')

torch.save({
    'generator': generator_wgan_gp.state_dict(),
    'critic': critic_wgan_gp.state_dict(),
    'history': history_wgan_gp,
}, 'wgan_gp_checkpoint.pth')

print("Models saved successfully!")
print("  - wgan_checkpoint.pth")
print("  - wgan_gp_checkpoint.pth")

## 9. Summary & Notes for Part C

### WGAN vs WGAN-GP Comparison

| Aspect | WGAN (Weight Clipping) | WGAN-GP (Gradient Penalty) |
|--------|------------------------|---------------------------|
| **Lipschitz Constraint** | Weight clipping $w \in [-c, c]$ | Gradient penalty on interpolates |
| **Training Stability** | Moderate (sensitive to clip value) | Better (no clipping artifacts) |
| **Sample Quality** | Good | Typically better |
| **FID Score** | Higher (worse) | Lower (better) |
| **Training Speed** | Faster per iteration | Slower (extra backward pass for GP) |

### Key Observations for Report (Part C)
1. **WGAN** may suffer from capacity underuse due to weight clipping limiting the critic's expressiveness.
2. **WGAN-GP** generally produces sharper, more diverse samples with lower FID.
3. Both models generate images unconditionally (no class conditioning).
4. FID scores should be compared against VAE ($\beta=1$) and VQ-VAE ($K=256$) from Part A.

### Files Generated
- `wgan_samples/epoch_XXX.png` — WGAN samples across epochs
- `wgan_gp_samples/epoch_XXX.png` — WGAN-GP samples across epochs
- `plots/wgan_100_samples.png` — 100 WGAN samples for Part C
- `plots/wgan_gp_100_samples.png` — 100 WGAN-GP samples for Part C
- `plots/training_losses.png` — Loss curves
- `wgan_checkpoint.pth` / `wgan_gp_checkpoint.pth` — Model weights